# Análisis Exploratorio de Datos (EDA)
## Sistema de Optimización Energética - Mega Plaza Chimbote

Este notebook realiza un análisis exploratorio completo del dataset de consumo energético.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurar estilo de visualización
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
%matplotlib inline

> **Nota Importante**: Este notebook usa el dataset personalizado `datos_retail_para_modelos.csv`.
> 
> Si aún no tienes el dataset:
> 1. Coloca tu archivo `datos_retail_para_modelos.csv` en `../../data/raw/`
> 2. Consulta `../../data/README.md` para ver la estructura requerida
> 3. Lee `../../docs/ENTRENAMIENTO.md` para instrucciones completas

## 1. Carga de Datos

In [ ]:
# Cargar dataset
data_path = "../../data/raw/datos_retail_para_modelos.csv"
df = pd.read_csv(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(f"Dataset cargado: {len(df)} registros")
df.head()

## 2. Información General del Dataset

In [ ]:
# Información general
print("=== Información del Dataset ===")
df.info()
print("
=== Primeras filas ===")
display(df.head(10))
print("
=== Estadísticas descriptivas ===")
display(df.describe())

In [ ]:
# Verificar valores nulos
print("=== Valores Nulos ===")
null_counts = df.isnull().sum()
null_percentage = (null_counts / len(df)) * 100
null_df = pd.DataFrame({"Nulos": null_counts, "Porcentaje": null_percentage})
display(null_df[null_df["Nulos"] > 0])

## 3. Análisis del Consumo Energético

In [ ]:
# Distribución del consumo
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df["consumo_energia"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Consumo (kWh)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución del Consumo Energético")
axes[0].grid(True, alpha=0.3)

axes[1].boxplot(df["consumo_energia"])
axes[1].set_ylabel("Consumo (kWh)")
axes[1].set_title("Boxplot del Consumo Energético")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Consumo promedio: {df["consumo_energia"].mean():.2f} kWh")
print(f"Consumo mínimo: {df["consumo_energia"].min():.2f} kWh")
print(f"Consumo máximo: {df["consumo_energia"].max():.2f} kWh")
print(f"Desviación estándar: {df["consumo_energia"].std():.2f} kWh")

## 4. Análisis Temporal

In [ ]:
# Consumo por hora del día
consumo_por_hora = df.groupby("hora_del_dia")["consumo_energia"].mean()

plt.figure(figsize=(12, 5))
plt.plot(consumo_por_hora.index, consumo_por_hora.values, marker="o", linewidth=2)
plt.xlabel("Hora del Día")
plt.ylabel("Consumo Promedio (kWh)")
plt.title("Consumo Energético por Hora del Día")
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24))
plt.show()

In [ ]:
# Consumo por día de la semana
dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
consumo_por_dia = df.groupby("dia_de_la_semana")["consumo_energia"].mean()

plt.figure(figsize=(10, 5))
plt.bar(range(7), consumo_por_dia.values, color="skyblue", edgecolor="black")
plt.xlabel("Día de la Semana")
plt.ylabel("Consumo Promedio (kWh)")
plt.title("Consumo Energético por Día de la Semana")
plt.xticks(range(7), dias, rotation=45)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 5. Análisis de Correlaciones

In [ ]:
# Matriz de correlación
columnas_numericas = ["consumo_energia", "pies_cuadrados", "temperatura_aire", 
                     "cobertura_nubes", "presion_nivel_mar", "velocidad_viento",
                     "hora_del_dia", "dia_de_la_semana", "es_fin_de_semana"]

correlacion = df[columnas_numericas].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title("Matriz de Correlación - Variables del Consumo Energético")
plt.tight_layout()
plt.show()

print("
Correlaciones con consumo_energia:")
print(correlacion["consumo_energia"].sort_values(ascending=False))

## 6. Relación Consumo vs Temperatura

In [ ]:
# Scatter plot: Consumo vs Temperatura
plt.figure(figsize=(10, 6))
plt.scatter(df["temperatura_aire"], df["consumo_energia"], alpha=0.3)
plt.xlabel("Temperatura del Aire (°C)")
plt.ylabel("Consumo Energético (kWh)")
plt.title("Relación entre Temperatura y Consumo Energético")
plt.grid(True, alpha=0.3)

# Agregar línea de tendencia
z = np.polyfit(df["temperatura_aire"], df["consumo_energia"], 1)
p = np.poly1d(z)
plt.plot(df["temperatura_aire"].sort_values(), 
         p(df["temperatura_aire"].sort_values()), 
         "r--", linewidth=2, label="Tendencia")
plt.legend()
plt.show()

## 7. Series Temporales

In [ ]:
# Serie temporal del consumo (primeros 1000 registros)
df_sample = df.head(1000).copy()
df_sample = df_sample.set_index("timestamp")

plt.figure(figsize=(15, 6))
plt.plot(df_sample.index, df_sample["consumo_energia"], linewidth=1)
plt.xlabel("Fecha y Hora")
plt.ylabel("Consumo (kWh)")
plt.title("Serie Temporal del Consumo Energético (Primeros 1000 registros)")
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Detección de Outliers

In [ ]:
# Detectar outliers usando IQR
Q1 = df["consumo_energia"].quantile(0.25)
Q3 = df["consumo_energia"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = df[(df["consumo_energia"] < lower_bound) | (df["consumo_energia"] > upper_bound)]

print(f"Número total de registros: {len(df)}")
print(f"Número de outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
print(f"Rango normal: [{lower_bound:.2f}, {upper_bound:.2f}] kWh")

if len(outliers) > 0:
    print("
Muestra de outliers:")
    display(outliers.head())

## 9. Conclusiones y Recomendaciones

### Conclusiones del Análisis Exploratorio:

1. **Patrones Temporales**: El consumo energético muestra patrones claros por hora del día y día de la semana.
2. **Temperatura**: Existe una relación entre la temperatura y el consumo energético.
3. **Outliers**: Se detectaron algunos valores atípicos que podrían requerir investigación adicional.
4. **Variables Relevantes**: Las variables más correlacionadas con el consumo son hora del día y temperatura.

### Recomendaciones:

1. Aplicar limpieza de outliers extremos antes del modelado.
2. Crear features temporales adicionales (lags, rolling means) para capturar tendencias.
3. Considerar la estacionalidad en los modelos de predicción.
4. Explorar técnicas de clustering para identificar patrones de consumo similares.